# Notebook C — Replay · Generalisation · Seed Repeats
**Project:** Mitigating Proximity-Structured Catastrophic Forgetting in LoRA Fine-Tuned Medical LLMs  
**Target:** BlackboxNLP @ EMNLP 2026  
**Account:** 3 of 3  
**Experiments:** E09–E11 (replay) · E12–E13 (generalisation on 2B model) · E14–E17 (seed repeats)  
**Estimated time:** ~23 GPU-hours across 3 Colab sessions  

---

## ✅ Prerequisites — read before running

1. **Notebook A must be complete.** It produces `E00_baseline.csv` which this notebook loads at Cell 2. Without it, Cell 2 will throw a `FileNotFoundError` immediately.
2. **Notebook A's E02 (r=16 MedQA) must be complete.** The H6 Spearman test at the end of Cell 10 reads E02 forgetting values from `all_results.csv`. You can still run E12–E13 without it but the H6 statistical test will show no data.
3. **Google Drive must be mounted.** All results are saved immediately after every experiment. If Colab disconnects, reconnect, re-run Cells 0–4, then skip to whichever experiment cell you need next — each cell has a resume guard that skips if results already exist.

---

## 📋 Steps to run this notebook

| Step | What to do |
|------|------------|
| 1 | Open this notebook in Google Colab |
| 2 | Go to **Runtime → Change runtime type → Hardware accelerator → T4 GPU** |
| 3 | Click **Runtime → Run all** OR run cells one by one from top to bottom |
| 4 | Cell 1 will pop up a Google Drive authorisation window — click Allow |
| 5 | Cell 2 will crash if Notebook A hasn't run yet — fix that first |
| 6 | Each experiment cell (6, 7, 8, 9, 10, 11) takes 2–4 hours. Run one per Colab session |
| 7 | After each session ends (timeout or disconnect), reconnect, re-run Cells 0–4, then continue |
| 8 | When all cells are done, hand `all_results.csv` from Drive to Notebook D for analysis |

---

## 🔄 After a Colab disconnect

Each experiment cell has a **resume guard** at the top. When you reconnect and re-run:
- Cells 0–4 take < 5 minutes (install + mount + load utilities)
- Jump to the cell you were running — the guard will print `✅ already complete` and skip if it finished, or re-run from scratch if it was interrupted mid-way
- **Never manually delete rows from `all_results.csv`** — the guard uses row count to detect completion

---

## ⚠️ Model notes — Qwen3.5 specific

- **Model:** `Qwen/Qwen3.5-9B` (primary) and `Qwen/Qwen3.5-2B` (small generalisation model)  
- **Thinking mode:** Qwen3.5 has a built-in `<think>...</think>` block that appears before answers. This is **disabled during MMLU evaluation** via `enable_thinking=False` in the chat template call. If not disabled, the regex parser would extract the first letter of the thinking block instead of the answer.  
- **During fine-tuning:** Thinking mode is left at its default (not explicitly disabled) so the model trains with its full instruction-following capability.  
- **Transformers version:** Must be ≥ 4.51.0 to include the `Qwen3_5ForCausalLM` class. Cell 0 pins this.


https://www.kaggle.com/code/akankshanarula/e10-e11-notebook

## Cell 0 — Install dependencies

Pins `transformers>=4.51.0` which is the minimum version that includes `Qwen3_5ForCausalLM`.  
Earlier versions will throw `ValueError: Unrecognized model` when loading Qwen3.5-9B.  
Run once per Colab session — takes about 2 minutes.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"   # makes CUDA errors synchronous & pinpoint-accurate

In [ ]:
# Install all dependencies with pinned compatible versions
# transformers 4.51.3 is the first stable PyPI release with Qwen3_5ForCausalLM.
# Pinning avoids pulling in 5.x dev/nightly builds which have broken qwen3_5 registration.
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.51.3",          # first stable release with Qwen3.5 support
    "peft>=0.12.0",
    "trl>=0.10.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.33.0",
    "datasets>=2.20.0",
    "sentence-transformers>=3.0.0",
    "scipy", "scikit-learn", "matplotlib", "seaborn",
])

print('✅ Dependencies installed')

# Verify transformers version
!pip uninstall transformers -y -q
!pip install "transformers[serving] @ git+https://github.com/huggingface/transformers.git@main" -q
!pip install --upgrade accelerate bitsandbytes peft trl datasets -q


In [ ]:
# Verify imports load correctly after install
# If this cell errors, restart the kernel and re-run from Cell 0
from transformers import AutoModelForCausalLM, AutoTokenizer
print(f'✅ transformers import OK')

import torch
print(f'   CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU            : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM           : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
# All installs already handled in Cell 0 above.
# This cell is kept as a placeholder so cell numbering stays consistent.
print('✅ Install cell skipped — already done above')


## Cell 1 — Mount Google Drive + configuration

All results are written to `lora_forgetting_research/results/all_results.csv` on your Drive **immediately** after every subject evaluation. This means a Colab crash loses at most one subject's worth of compute (~2 minutes), not an entire experiment run.

**Model IDs used in this notebook:**
- `Qwen/Qwen3.5-9B` — primary 9B instruction-tuned model (Qwen3.5 series, February 2026)
- `Qwen/Qwen3.5-2B` — small model for H6 generalisation test (replaces original 1.7B plan; 2B is the closest available Qwen3.5 size)

The 9B model in 4-bit NF4 quantisation uses ~5.5 GB VRAM, comfortably fitting on the free T4's 15 GB.


In [ ]:
import os, torch, random, re
import numpy as np
import pandas as pd
from datetime import datetime
import gc

# ── Kaggle paths ─────────────────────────────────────────────────────────────
# /kaggle/working  → writable scratch (ALSO used for download-ready outputs)
# /kaggle/output   → persisted as dataset output (attach in "Output" panel)
KAGGLE_WORKING   = "/kaggle/working"
KAGGLE_OUTPUT    = "/kaggle/output"

BASE_DIR     = KAGGLE_WORKING                       # primary scratch space
RESULTS_DIR  = f"{KAGGLE_OUTPUT}/results"           # goes to output dataset
ADAPTERS_DIR = f"{KAGGLE_OUTPUT}/adapters"

# Mirror dir under /kaggle/working so results are always downloadable
WORKING_RESULTS_DIR = f"{KAGGLE_WORKING}/lora_forgetting_research/results"
WORKING_ADAPTERS_DIR = f"{KAGGLE_WORKING}/lora_forgetting_research/adapters"

RESULTS_CSV      = f"{RESULTS_DIR}/all_results.csv"
RESULTS_CSV_WORK = f"{WORKING_RESULTS_DIR}/all_results.csv"   # always-accessible copy

BASELINE_CSV = f"/kaggle/input/datasets/akankshanarula/lora-forgetting-exp/E00_baseline.csv"

os.makedirs(RESULTS_DIR,       exist_ok=True)
os.makedirs(ADAPTERS_DIR,      exist_ok=True)
os.makedirs(WORKING_RESULTS_DIR,  exist_ok=True)
os.makedirs(WORKING_ADAPTERS_DIR, exist_ok=True)

# ── Model IDs (Qwen3.5 series) ───────────────────────────────────────────────
MODEL_9B   = "Qwen/Qwen3.5-9B"
MODEL_2B   = "Qwen/Qwen3.5-2B"
MODEL_8B   = MODEL_9B       # alias for shared code compatibility
MODEL_17B  = MODEL_2B       # alias for shared code compatibility

# ── Seeds ────────────────────────────────────────────────────────────────────
SEED_PRIMARY = 42
SEED_REPEAT  = 7

# ── MMLU subjects ────────────────────────────────────────────────────────────
MMLU_SUBJECTS = [
    "abstract_algebra","anatomy","astronomy","business_ethics",
    "clinical_knowledge","college_biology","college_chemistry",
    "college_computer_science","college_mathematics","college_medicine",
    "college_physics","computer_security","conceptual_physics",
    "econometrics","electrical_engineering","elementary_mathematics",
    "formal_logic","global_facts","high_school_biology",
    "high_school_chemistry","high_school_computer_science",
    "high_school_european_history","high_school_geography",
    "high_school_government_and_politics","high_school_macroeconomics",
    "high_school_mathematics","high_school_microeconomics",
    "high_school_physics","high_school_psychology","high_school_statistics",
    "high_school_us_history","high_school_world_history","human_aging",
    "human_sexuality","international_law","jurisprudence","logical_fallacies",
    "machine_learning","management","marketing","medical_genetics",
    "miscellaneous","moral_disputes","moral_scenarios","nutrition",
    "philosophy","prehistory","professional_accounting","professional_law",
    "professional_medicine","professional_psychology","public_relations",
    "security_studies","sociology","us_foreign_policy","virology",
    "world_religions"
]

MEDICAL_MMLU = {
    "anatomy","clinical_knowledge","college_biology","college_medicine",
    "high_school_biology","medical_genetics","professional_medicine",
    "virology","human_aging"
}

DOMAIN_DESCRIPTIONS = {
    "medqa":      "medicine clinical knowledge anatomy pharmacology pathology diagnosis treatment disease symptoms",
    "gsm8k":      "mathematics arithmetic algebra word problems numerical computation",
    "codealpaca": "programming code software functions algorithms debugging python",
}

# ── Seed helper ───────────────────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED_PRIMARY)

print(f"✅ Config loaded")
print(f"   Primary model  : {MODEL_9B}")
print(f"   Small model    : {MODEL_2B}")
print(f"   Results dir    : {RESULTS_DIR}")
print(f"   Adapters dir   : {ADAPTERS_DIR}")
print(f"   CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU            : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM           : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# ── Periodic download helper ──────────────────────────────────────────────────
# Call download_results() at any time to create a zip of current results
# in /kaggle/working — then click the file in the Kaggle output panel to download.
import shutil, zipfile

def download_results(tag="checkpoint"):
    """Zip current results and save to /kaggle/working for manual download."""
    zip_path = f"{KAGGLE_WORKING}/results_{tag}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(WORKING_RESULTS_DIR):
            for file in files:
                fp = os.path.join(root, file)
                arcname = os.path.relpath(fp, KAGGLE_WORKING)
                zf.write(fp, arcname)
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"✅ Results zipped to: {zip_path}  ({size_mb:.2f} MB)")
    print(f"   Download via Kaggle's Output panel → /kaggle/working/results_{tag}.zip")
    return zip_path

print("✅ Download helper ready — call download_results('after_E09') etc. after each experiment")
print(f"   Results mirror at: {WORKING_RESULTS_DIR}")


## Cell 2 — Load E00 baseline from Notebook A

This cell will **raise a FileNotFoundError** if Notebook A has not completed E00 yet.  
The baseline CSV contains the per-subject accuracy of `Qwen3.5-9B` before any fine-tuning.  
All forgetting values in this notebook are computed as `post_accuracy - baseline_accuracy` per subject.


In [ ]:
if not os.path.exists(BASELINE_CSV):
    raise FileNotFoundError(
        f"❌ Baseline not found: {BASELINE_CSV}\n"
        f"   You must run Notebook A (E00) first!"
    )

baseline_df   = pd.read_csv(BASELINE_CSV)
baseline_accs = dict(zip(baseline_df["subject"], baseline_df["accuracy"]))

print(f"✅ Baseline loaded: {len(baseline_accs)} subjects")
print(f"   Mean baseline accuracy: {np.nanmean(list(baseline_accs.values())):.3f}")
print(f"   Medical subjects mean : {np.nanmean([baseline_accs[s] for s in MEDICAL_MMLU if s in baseline_accs]):.3f}")
print(f"   Non-medical mean      : {np.nanmean([v for s,v in baseline_accs.items() if s not in MEDICAL_MMLU]):.3f}")


## Cell 3 — Shared utility functions

Contains five functions used by every experiment cell:

- **`log_result`** — appends one row to `all_results.csv` on Drive immediately. Never batches.
- **`load_model_4bit`** — loads any model in 4-bit NF4 quantisation (QLoRA-style). The 9B model takes ~5.5 GB VRAM.
- **`eval_mmlu_subject`** — evaluates the model on one MMLU subject. ⚠️ Contains the Qwen3.5 thinking-mode fix: `enable_thinking=False` is passed to `apply_chat_template` so the model answers directly without emitting a `<think>...</think>` block that would break the letter-extraction regex.
- **`eval_all_mmlu`** — loops over all 57 subjects, printing progress every 5.
- **`run_finetune`** — applies LoRA to the model and trains with SFTTrainer. Thinking mode is NOT disabled here — the model fine-tunes with full capability.
- **`log_experiment_results`** — helper that writes per-subject forgetting rows in one call.
- **`run_proximity_test`** — runs H1 (Pearson r) and H3 (t-test direction) after each experiment.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset, Dataset, concatenate_datasets
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, ttest_ind, spearmanr

# ── Result logger — saves immediately, never loses data ──────────────────────
def log_result(exp_id, result_dict):
    row = {"exp_id": exp_id, "timestamp": datetime.now().isoformat(), **result_dict}
    # Keep only the 13 standard columns to avoid CSV column-count mismatch
    # (proximity rows include extra stats that would corrupt the CSV)
    STANDARD_COLS = ["exp_id","timestamp","model","domain","lora_rank","n_steps",
                     "replay_size","seed","subject","is_medical","accuracy","delta_acc","forgetting"]
    row_clean = {k: row.get(k, "") for k in STANDARD_COLS}
    df_new = pd.DataFrame([row_clean])
    # Write to output dir
    if os.path.exists(RESULTS_CSV):
        df_new.to_csv(RESULTS_CSV, mode="a", header=False, index=False)
    else:
        df_new.to_csv(RESULTS_CSV, index=False)
    # Mirror to /kaggle/working so it's always downloadable
    if os.path.exists(RESULTS_CSV_WORK):
        df_new.to_csv(RESULTS_CSV_WORK, mode="a", header=False, index=False)
    else:
        df_new.to_csv(RESULTS_CSV_WORK, index=False)
    # Also store proximity stats separately in a dedicated file
    if any(k not in STANDARD_COLS for k in row.keys()):
        prox_path = RESULTS_CSV.replace("all_results.csv", "proximity_stats.csv")
        prox_work = RESULTS_CSV_WORK.replace("all_results.csv", "proximity_stats.csv")
        df_prox = pd.DataFrame([row])
        if os.path.exists(prox_path):
            df_prox.to_csv(prox_path, mode="a", header=False, index=False)
            df_prox.to_csv(prox_work, mode="a", header=False, index=False)
        else:
            df_prox.to_csv(prox_path, index=False)
            df_prox.to_csv(prox_work, index=False)
    print(f"  💾 {exp_id} saved")

def safe_read_results():
    """Read results CSV robustly — skip any malformed rows."""
    if not os.path.exists(RESULTS_CSV):
        return pd.DataFrame()
    try:
        return pd.read_csv(RESULTS_CSV)
    except Exception:
        return pd.read_csv(RESULTS_CSV, on_bad_lines="skip")

# ── 4-bit model loader ───────────────────────────────────────────────────────
def load_model_4bit(model_id):
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb, device_map="auto",
        trust_remote_code=True
    )
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    model.config.use_cache = False
    print(f"✅ Loaded {model_id}")
    return model, tok

# ── MMLU evaluator — CHANGE 2: enable_thinking=False for Qwen3.5 ─────────────
# Qwen3.5 has a built-in chain-of-thought "thinking mode" that emits a
# <think>...</think> block BEFORE the final answer. If not disabled, the regex
# re.search(r"[ABCD]") would match the first letter inside the thinking block
# (e.g. "A blood test...") instead of the answer letter. enable_thinking=False
# suppresses this block entirely so the model answers directly.
def eval_mmlu_subject(model, tok, subject, max_samples=None):
    try:
        ds = load_dataset("cais/mmlu", subject, split="test", trust_remote_code=True)
    except Exception as e:
        print(f"  ⚠️ {subject}: {e}")
        return float("nan")
    if max_samples:
        ds = ds.select(range(min(max_samples, len(ds))))
    correct, total = 0, 0
    label_map = {0: "A", 1: "B", 2: "C", 3: "D"}
    for ex in ds:
        choices_str = "\n".join([f"{l}) {c}" for l, c in zip("ABCD", ex["choices"])])
        prompt = (
            "The following is a multiple choice question. "
            "Answer with only the letter A, B, C, or D.\n\n"
            f"Question: {ex['question']}\n{choices_str}\n\nAnswer:"
        )
        messages = [{"role": "user", "content": prompt}]
        try:
            # enable_thinking=False: disables <think>...</think> prefix (Qwen3.5 fix)
            text = tok.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False   # CRITICAL for Qwen3.5
            )
        except TypeError:
            # Fallback for tokenisers that don't support the kwarg yet
            try:
                messages_nothink = [{"role": "user", "content": prompt + " /no_think"}]
                text = tok.apply_chat_template(
                    messages_nothink, tokenize=False, add_generation_prompt=True
                )
            except Exception:
                text = prompt
        ids = tok(text, return_tensors="pt").input_ids.to(model.device)
        with torch.no_grad():
            out = model.generate(
                ids, max_new_tokens=5, do_sample=False,
                pad_token_id=tok.eos_token_id,
                eos_token_id=tok.eos_token_id
            )
        gen_text = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip().upper()
        match = re.search(r"[ABCD]", gen_text)
        pred = match.group(0) if match else "X"
        correct += int(pred == label_map[ex["answer"]])
        total += 1
    return correct / total if total > 0 else float("nan")

def eval_all_mmlu(model, tok, max_samples=None):
    results = {}
    for i, subj in enumerate(MMLU_SUBJECTS):
        acc = eval_mmlu_subject(model, tok, subj, max_samples)
        results[subj] = acc
        if (i + 1) % 5 == 0 or (i + 1) == len(MMLU_SUBJECTS):
            print(f"  [{i+1:2d}/57] {subj}: {acc:.3f}")
    return results

# ── LoRA fine-tuner — CHANGE 3: thinking mode left at default during SFT ─────
# We do NOT disable thinking during training. The model should learn from its
# full reasoning capability. enable_thinking only affects eval.
def run_finetune(base_model_id, train_dataset, exp_id,
                 lora_rank=16, n_steps=500, seed=42):
    set_seed(seed)
    adapter_path = f"{ADAPTERS_DIR}/{exp_id}"
    os.makedirs(adapter_path, exist_ok=True)
    print(f"\n🔧 {exp_id} | rank={lora_rank} | steps={n_steps} | seed={seed}")
    model, tok = load_model_4bit(base_model_id)
    lora_cfg = LoraConfig(
        r=lora_rank, lora_alpha=lora_rank * 2,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05, bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    trainer = SFTTrainer(
        model=model,
        processing_class=tok,        # ← change this line only
        train_dataset=train_dataset,
        args=SFTConfig(
            output_dir=adapter_path,
            max_steps=n_steps,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,          # effective batch = 8
            warmup_steps=min(50, n_steps // 10),
            learning_rate=2e-4,
            bf16=True,
            fp16=False,
            gradient_checkpointing=False,           # ← MUST be False for Mamba/SSM
            optim="paged_adamw_8bit",               # optimizer lives mostly off-GPU
            max_length=256,                         # shorter seqs = less activation RAM
            dataset_text_field="text",
            logging_steps=50,
            save_strategy="steps",
            save_steps=100,
            save_total_limit=2,
            seed=seed,
            report_to="none",
            dataloader_pin_memory=False,
        ),
    )
    print(f"🚀 Training {n_steps} steps ...")
    trainer.train()
    model.save_pretrained(adapter_path)
    tok.save_pretrained(adapter_path)
    print(f"💾 Adapter saved: {adapter_path}")
    return model, tok

# ── Bulk result logger helper ────────────────────────────────────────────────
def log_experiment_results(exp_id, post_accs, model_id, domain,
                            rank, n_steps, replay_size, seed,
                            forgetting_dict=None):
    for subj in MMLU_SUBJECTS:
        base_acc = baseline_accs.get(subj, float("nan"))
        post_acc = post_accs.get(subj, float("nan"))
        delta    = post_acc - base_acc
        if forgetting_dict is not None:
            forgetting_dict[subj] = delta
        log_result(exp_id, {
            "model":       model_id,
            "domain":      domain,
            "lora_rank":   rank,
            "n_steps":     n_steps,
            "replay_size": replay_size,
            "seed":        seed,
            "subject":     subj,
            "accuracy":    round(post_acc, 4) if not np.isnan(post_acc) else "nan",
            "is_medical":  subj in MEDICAL_MMLU,
            "delta_acc":   round(delta, 4)    if not np.isnan(delta)    else "nan",
            "forgetting":  round(-delta, 4)   if not np.isnan(delta)    else "nan",
        })
    return forgetting_dict

# ── Proximity test (H1 + H3) ─────────────────────────────────────────────────
_embedder = None
def get_embedder():
    global _embedder
    if _embedder is None:
        _embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    return _embedder

def run_proximity_test(domain_key, forgetting_dict, exp_id):
    """
    H1: Pearson r between cosine_sim(subject, domain) and forgetting magnitude.
    H3: Two-sample t-test — proximal (medical) subjects vs distal (non-medical).
    The DIRECTION of H3 (proximal > or < distal) is the key result.
    """
    emb = get_embedder()
    domain_vec = emb.encode(DOMAIN_DESCRIPTIONS[domain_key], normalize_embeddings=True)
    subj_texts = [s.replace("_", " ") for s in MMLU_SUBJECTS]
    subj_vecs  = emb.encode(subj_texts, normalize_embeddings=True,
                             batch_size=32, show_progress_bar=False)
    sims = {s: float(np.dot(domain_vec, v))
            for s, v in zip(MMLU_SUBJECTS, subj_vecs)}
    valid = [s for s in MMLU_SUBJECTS
             if s in forgetting_dict and not np.isnan(forgetting_dict[s])]
    sv = [sims[s] for s in valid]
    fv = [forgetting_dict[s] for s in valid]
    r, p = pearsonr(sv, fv)
    prox = [forgetting_dict[s] for s in valid if s in MEDICAL_MMLU]
    dist = [forgetting_dict[s] for s in valid if s not in MEDICAL_MMLU]
    t_stat, t_p = (ttest_ind(prox, dist)
                   if len(prox) > 1 and len(dist) > 1
                   else (float("nan"), float("nan")))
    pm, dm = np.mean(prox), np.mean(dist)
    # H3 direction: negative delta = forgetting (lower = more forgotten)
    # proximal mean < distal mean → proximal subjects forget MORE → matches InternAL
    direction = ("proximal>distal (matches InternAL NeurIPS 2025)"
                 if pm < dm else "proximal<=distal (reversal of InternAL)")
    result = {
        "domain": domain_key,
        "h1_pearson_r":                 round(r, 4),
        "h1_pearson_p":                 round(p, 6),
        "h3_t_stat":                    round(t_stat, 4) if not np.isnan(t_stat) else "nan",
        "h3_t_p":                       round(t_p,    6) if not np.isnan(t_p)    else "nan",
        "h3_proximal_mean_forgetting":  round(pm, 4) if not np.isnan(pm) else "nan",
        "h3_distal_mean_forgetting":    round(dm, 4) if not np.isnan(dm) else "nan",
        "h3_direction": direction,
        "n_subjects": len(valid),
    }
    print(f"  H1 r={r:.3f} p={p:.5f} | H3: {direction}")
    print(f"     proximal mean={pm:.4f} | distal mean={dm:.4f}")
    return result

print("✅ All utilities ready")


## Cell 4 — Load MedQA training data

Loads 4,000 examples from the MedQA-USMLE training split, stratified by question category.  
Stratification is important for H1: if we over-represent one MedQA category (e.g. internal medicine),  
the domain embedding won't accurately represent the fine-tuning distribution, biasing the cosine similarity test.  

The loader handles two HuggingFace dataset formats automatically (column names changed between versions).  
This cell also pre-builds the two real MMLU replay buffers (50 and 100 examples) from the MMLU validation split,  
which is separate from the test split used for evaluation — so there is no data leakage.


In [ ]:
def get_stratified_medqa(n_total=4000, seed=42):
    print("⬇️  Loading MedQA-USMLE-4-options ...")
    ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="train",
                      trust_remote_code=True)
    df = ds.to_pandas()

    def get_options(row):
        if "option_0" in df.columns:
            return [row[f"option_{i}"] for i in range(4)]
        elif "options" in df.columns:
            opts = row["options"]
            if isinstance(opts, dict):
                return [opts[k] for k in sorted(opts.keys())[:4]]
            elif isinstance(opts, list):
                return opts[:4]
        return ["A", "B", "C", "D"]

    if "meta_info" in df.columns and df["meta_info"].nunique() > 1:
        n_per_cat = max(500, n_total // df["meta_info"].nunique())
        sampled = df.groupby("meta_info", group_keys=False).apply(
            lambda x: x.sample(min(len(x), n_per_cat), random_state=seed))
    else:
        sampled = df.sample(min(n_total, len(df)), random_state=seed)

    sampled = sampled.sample(frac=1, random_state=seed).reset_index(drop=True)
    rows = []
    for _, row in sampled.iterrows():
        opts = get_options(row)
        opts_str = "\n".join([f"{l}) {c}" for l, c in zip("ABCD", opts)])
        answer = str(row.get("answer", "A")).strip().upper()
        if answer not in "ABCD": answer = "A"
        rows.append({"text": f"Question: {row['question']}\n{opts_str}\nAnswer: {answer}"})

    result = Dataset.from_list(rows)
    print(f"✅ MedQA: {len(result)} examples")
    return result


def build_real_mmlu_replay(n_examples: int, seed: int = 42) -> Dataset:
    """
    Build a real MMLU replay buffer from the MMLU *validation* split.
    Using validation (not test) avoids contaminating the evaluation metric.
    Samples from 20 randomly chosen subjects for diversity.
    """
    print(f"⬇️  Building real MMLU replay buffer ({n_examples} examples) ...")
    all_rows = []
    per_subject = max(1, n_examples // min(10, len(MMLU_SUBJECTS)))
    subjects_sample = random.Random(seed).sample(MMLU_SUBJECTS, min(20, len(MMLU_SUBJECTS)))

    for subj in subjects_sample:
        try:
            ds = load_dataset("cais/mmlu", subj, split="validation",
                              trust_remote_code=True)
            if len(ds) == 0:
                ds = load_dataset("cais/mmlu", subj, split="dev",
                                  trust_remote_code=True)
            label_map = {0: "A", 1: "B", 2: "C", 3: "D"}
            for ex in ds.select(range(min(per_subject, len(ds)))):
                choices_str = "\n".join(
                    [f"{l}) {c}" for l, c in zip("ABCD", ex["choices"])]
                )
                answer = label_map[ex["answer"]]
                all_rows.append({
                    "text": (f"Question: {ex['question']}\n"
                             f"{choices_str}\nAnswer: {answer}")
                })
        except Exception as e:
            print(f"  ⚠️ {subj}: {e}")
            continue

    random.Random(seed).shuffle(all_rows)
    all_rows = all_rows[:n_examples]
    result = Dataset.from_list(all_rows)
    print(f"✅ Real replay buffer: {len(result)} examples")
    return result


# Load training data and pre-build replay buffers
medqa_train     = get_stratified_medqa(n_total=4000, seed=SEED_PRIMARY)
replay_50_real  = build_real_mmlu_replay(50,  seed=SEED_PRIMARY)
replay_100_real = build_real_mmlu_replay(100, seed=SEED_PRIMARY)

print(f"\n✅ Training data ready")
print(f"   MedQA train  : {len(medqa_train)} examples")
print(f"   Replay (50)  : {len(replay_50_real)} examples")
print(f"   Replay (100) : {len(replay_100_real)} examples")


---
## Experiments E09–E11 — Replay (H5)

**Hypothesis H5:** A small MMLU replay buffer (mixed into training) reduces mean forgetting by ≥30%  
with <1% medical task performance cost, and self-generated replay matches real replay.

**What is replay?**  
Replay (also called rehearsal) is a continual learning technique where you mix a small number of  
examples from *old* knowledge into the *new* training data. This forces the model to rehearse what it  
already knows while learning the new domain, preventing the gradient updates from overwriting it.

**Three conditions:**
| Experiment | Replay data | Size | Tests |
|---|---|---|---|
| E09 | Real MMLU validation examples | 50 | Minimum buffer size |
| E10 | Real MMLU validation examples | 100 | **Main replay condition** |
| E11 | Self-generated by the model itself | 100 | Ding et al. 2026 *Talking to Yourself* — can the model protect its own knowledge without stored data? |

**The clinical significance of E11:** If self-generated replay works, hospitals deploying medical AI  
do not need to store any original training data (which may be sensitive patient records)  
to prevent the fine-tuned model from forgetting general medical knowledge.

⏱️ Each experiment: ~2.5 hours. Run one per Colab session.


### Cell 5 — Self-generated replay builder (used by E11)

Loads the base model (before any fine-tuning) and asks it to generate 100 multiple-choice  
questions about non-medical MMLU topics (math, history, law, etc.).  

Why non-medical topics? Because the goal of replay is to anchor *general* knowledge that is at  
risk of being overwritten by medical fine-tuning. Medical topics are the training domain —  
they do not need anchoring.

⚠️ Note: `enable_thinking=False` is also used here during generation so the model produces  
clean Q&A format without a leading thinking block that would corrupt the format validation regex.


In [ ]:
def build_self_generated_replay(model, tokenizer, n_examples: int = 100,
                                 seed: int = 42) -> Dataset:
    """
    Tests Ding et al. 2026 (Talking to Yourself): ask the model to generate
    its own rehearsal Q&A on non-medical topics and use those as replay data.
    enable_thinking=False ensures clean Q&A format without <think> prefix.
    """
    print(f"⬇️  Generating self-generated replay ({n_examples} examples) ...")
    set_seed(seed)

    non_medical_subjects = [s for s in MMLU_SUBJECTS if s not in MEDICAL_MMLU]
    topic_sample = random.sample(non_medical_subjects, min(20, len(non_medical_subjects)))

    prompt_template = (
        "Generate a multiple choice question about {topic} with 4 options (A, B, C, D) "
        "and provide the correct answer. Format exactly as:\n"
        "Question: [your question]\nA) [option]\nB) [option]\nC) [option]\nD) [option]\n"
        "Answer: [A/B/C/D]\n"
    )

    all_rows = []
    per_topic = max(1, n_examples // len(topic_sample))
    model.eval()

    for topic in topic_sample:
        topic_clean = topic.replace("_", " ")
        prompt = prompt_template.format(topic=topic_clean)
        messages = [{"role": "user", "content": prompt}]
        try:
            # enable_thinking=False: prevents <think>...</think> from corrupting
            # the generated Q&A format validation check below
            text = tokenizer.apply_chat_template(
                messages, tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False
            )
        except TypeError:
            try:
                messages_nothink = [{"role": "user",
                                     "content": prompt + " /no_think"}]
                text = tokenizer.apply_chat_template(
                    messages_nothink, tokenize=False,
                    add_generation_prompt=True
                )
            except Exception:
                text = prompt

        for _ in range(per_topic):
            ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
            with torch.no_grad():
                out = model.generate(
                    ids, max_new_tokens=200, do_sample=True,
                    temperature=0.8, top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id
                )
            generated = tokenizer.decode(
                out[0, ids.shape[1]:], skip_special_tokens=True
            )
            # Only keep well-formed Q&A blocks
            if "Question:" in generated and re.search(r"Answer:\s*[ABCD]", generated):
                lines = [l for l in generated.strip().split("\n") if l.strip()]
                all_rows.append({"text": "\n".join(lines[:7])})

        if len(all_rows) >= n_examples:
            break

    all_rows = all_rows[:n_examples]
    random.Random(seed).shuffle(all_rows)
    result = Dataset.from_list(all_rows)
    print(f"✅ Self-generated replay: {len(result)} examples from {len(topic_sample)} topics")
    return result

print("✅ Self-generated replay builder ready")


### Cell 6 — E09: 50 real MMLU replay examples

Fine-tunes Qwen3.5-9B on MedQA + 50 real MMLU validation examples (combined and shuffled).  
The 50 examples are drawn from 20 different MMLU subjects for diversity.  
This is the minimum buffer size condition — does even 50 examples protect general knowledge?

⏱️ ~2.5 hours. **Safe to disconnect after this cell completes.**


### Cell 7 — E10: 100 real MMLU replay examples ★ Main replay condition

This is the primary replay experiment. 100 real MMLU validation examples mixed into MedQA training.  
Compare this against E02 (no replay, from Notebook A) to measure how much forgetting replay prevents.  
Compare against E11 (self-generated) to test the Talking to Yourself 2026 claim.

⏱️ ~2.5 hours. **Safe to disconnect after this cell completes.**


In [ ]:
# ── Resume guard ─────────────────────────────────────────────────────────────
_skip_E10 = False
if os.path.exists(RESULTS_CSV):
    _df_chk = safe_read_results()
    _done = _df_chk[(_df_chk["exp_id"]=="E10") & (_df_chk["subject"].isin(MMLU_SUBJECTS))]
    if len(_done) >= 50:
        print(f"✅ E10 already complete ({len(_done)} subjects). Skipping.")
        _skip_E10 = True

if not _skip_E10:
    print("\n" + "="*60)
    print("▶  E10: MedQA + 100 real MMLU replay, rank=16, steps=500  [MAIN]")
    print("="*60)

    medqa_plus_100 = concatenate_datasets([medqa_train, replay_100_real]).shuffle(seed=SEED_PRIMARY)
    print(f"   Training set size: {len(medqa_plus_100)}")

    ft_model_E10, ft_tok_E10 = run_finetune(
        MODEL_9B, medqa_plus_100, "E10", lora_rank=16, n_steps=500, seed=SEED_PRIMARY)
    ft_model_E10.eval()

    post_accs_E10 = eval_all_mmlu(ft_model_E10, ft_tok_E10)
    fgt_E10 = {}
    log_experiment_results("E10", post_accs_E10, MODEL_9B,
                           "medqa", 16, 500, 100, SEED_PRIMARY, fgt_E10)

    prox_E10 = run_proximity_test("medqa", fgt_E10, "E10")
    log_result("E10_proximity", {
        "model": MODEL_9B, "domain": "medqa", "lora_rank": 16,
        "n_steps": 500, "replay_size": 100, "seed": SEED_PRIMARY,
        "subject": "ALL_PROXIMITY_STATS",
        **prox_E10, "delta_acc": 0, "forgetting": 0,
        "accuracy": 0, "is_medical": False,
    })

    mean_fgt_E10 = -np.nanmean(list(fgt_E10.values()))
    print(f"\n📋 E10 SUMMARY")
    print(f"   Mean forgetting : {mean_fgt_E10:+.4f}")
    print(f"   H3 direction    : {prox_E10['h3_direction']}")
    print(f"   💾 Results saved → {RESULTS_CSV}")
    print(f"   ✅ E10 COMPLETE. Safe to disconnect.")

    download_results("after_E10")

    del ft_model_E10, ft_tok_E10
    torch.cuda.empty_cache(); gc.collect()


### Cell 8 — E11: 100 self-generated replay examples

Tests **Contradiction #5** from the literature map: Ding et al. 2026 (*Talking to Yourself*)  
vs the standard rehearsal assumption that real data is required.

This cell does three things in sequence:
1. Loads the **base model** (no fine-tuning) and generates 100 MMLU-style Q&A pairs on non-medical topics
2. Saves a sample of the generated data to Drive so you can inspect quality
3. Fine-tunes on MedQA + self-generated data

If E11's forgetting ≈ E10's forgetting, self-generated replay is sufficient.  
This has a major practical implication: hospitals don't need to store any training data to prevent forgetting.

⏱️ ~3 hours (generation adds ~20 min on top of the usual ~2.5 hours). **Safe to disconnect after.**


In [ ]:
# ── Resume guard ─────────────────────────────────────────────────────────────
_skip_E11 = False
if os.path.exists(RESULTS_CSV):
    _df_chk = safe_read_results()
    _done = _df_chk[(_df_chk["exp_id"]=="E11") & (_df_chk["subject"].isin(MMLU_SUBJECTS))]
    if len(_done) >= 50:
        print(f"✅ E11 already complete ({len(_done)} subjects). Skipping.")
        _skip_E11 = True

if not _skip_E11:
    print("\n" + "="*60)
    print("▶  E11: MedQA + 100 self-generated replay, rank=16, steps=500")
    print("   [Tests Contradiction #5: Ding et al. 2026 Talking to Yourself]")
    print("="*60)

    # Step 1: load BASE model to generate replay data
    print("⬇️  Loading base model for self-generation ...")
    temp_model, temp_tok = load_model_4bit(MODEL_9B)
    temp_model.eval()

    replay_100_selfgen = build_self_generated_replay(
        temp_model, temp_tok, n_examples=100, seed=SEED_PRIMARY)

    # Step 2: free base model before fine-tuning
    del temp_model, temp_tok
    torch.cuda.empty_cache(); gc.collect()
    print("🗑️  Base model freed — VRAM ready for fine-tuning")

    # Step 3: save sample to Drive for manual inspection
    selfgen_path = f"{RESULTS_DIR}/E11_selfgen_replay_samples.csv"
    pd.DataFrame(replay_100_selfgen).head(20).to_csv(selfgen_path, index=False)
    print(f"💾 20 sample self-gen questions saved: {selfgen_path}")
    print("   (Check this file to verify generation quality before trusting E11 results)")

    # Step 4: fine-tune on MedQA + self-generated replay
    medqa_plus_selfgen = concatenate_datasets(
        [medqa_train, replay_100_selfgen]).shuffle(seed=SEED_PRIMARY)
    print(f"   Training set size: {len(medqa_plus_selfgen)}")

    ft_model_E11, ft_tok_E11 = run_finetune(
        MODEL_9B, medqa_plus_selfgen, "E11",
        lora_rank=16, n_steps=500, seed=SEED_PRIMARY)
    ft_model_E11.eval()

    post_accs_E11 = eval_all_mmlu(ft_model_E11, ft_tok_E11)
    fgt_E11 = {}
    log_experiment_results("E11", post_accs_E11, MODEL_9B,
                           "medqa_selfgen", 16, 500, 100, SEED_PRIMARY, fgt_E11)

    prox_E11 = run_proximity_test("medqa", fgt_E11, "E11")
    log_result("E11_proximity", {
        "model": MODEL_9B, "domain": "medqa_selfgen", "lora_rank": 16,
        "n_steps": 500, "replay_size": 100, "seed": SEED_PRIMARY,
        "subject": "ALL_PROXIMITY_STATS",
        **prox_E11, "delta_acc": 0, "forgetting": 0,
        "accuracy": 0, "is_medical": False,
    })

    mean_fgt_E11 = -np.nanmean(list(fgt_E11.values()))
    print(f"\n📋 E11 SUMMARY")
    print(f"   Mean forgetting : {mean_fgt_E11:+.4f}")
    print(f"   H3 direction    : {prox_E11['h3_direction']}")

    # ── H5 comparison table ──────────────────────────────────────────────────
    print("\n" + "="*60)
    print("📊 H5 — Replay comparison (E09 vs E10 vs E11)")
    print("="*60)
    print(f"  Compare all three against E02 (no replay) from Notebook A")
    print(f"  {'Exp':6} {'Replay type':28} {'Mean fgt':12}")
    print(f"  {'-'*50}")
    df_h5 = safe_read_results()
    for eid, rtype in [("E09","50 real examples"),("E10","100 real examples"),("E11","100 self-generated")]:
        df_e = df_h5[df_h5["exp_id"]==eid].copy()
        df_e["f"] = pd.to_numeric(df_e["forgetting"],errors="coerce")
        df_e = df_e.dropna(subset=["f"])
        if len(df_e)>0:
            print(f"  {eid:6} {rtype:28} {df_e['f'].mean():+12.4f}")
    print(f"  (E02 no-replay result is in Notebook A's all_results.csv)")
    print(f"  → If E11 ≈ E10: self-gen replay matches real (Ding et al. validated in medical domain)")
    print(f"  💾 Results saved → {RESULTS_CSV}")
    print(f"  ✅ E11 COMPLETE. Safe to disconnect.")

    download_results("after_E11")

    del ft_model_E11, ft_tok_E11
    torch.cuda.empty_cache(); gc.collect()


---
## Experiments E12–E13 — Generalisation (H6)

**Hypothesis H6:** The per-subject forgetting pattern found on Qwen3.5-9B replicates  
on Qwen3.5-2B — same subjects are most affected, same direction, just smaller magnitudes.

**Why this matters:** If the pattern is model-size invariant (Spearman ρ > 0.70 between the  
two models' per-subject forgetting rankings), then the finding applies to the entire Qwen3.5  
model family, not just the 9B size. This is what lets you make a general claim rather than  
a single-model result.

**Model used:** `Qwen/Qwen3.5-2B` — the closest available Qwen3.5 size to the original 1.7B plan.  
The 2B model in 4-bit NF4 uses only ~1.5 GB VRAM and evaluates ~3× faster than the 9B model.

| Experiment | Model | Rank | Purpose |
|---|---|---|---|
| E12 | Qwen3.5-2B | r=16 | Primary generalisation condition — compare with E02 (9B, r=16) |
| E13 | Qwen3.5-2B | r=4  | Check if rank effect also replicates on smaller model |

⏱️ ~2.5 GPU-hours total for both. Can run in one session.


### Cell 9 — E12: Qwen3.5-2B baseline + fine-tune (r=16)

This cell does two things:
1. Runs the 2B model on all 57 MMLU subjects **before** fine-tuning (the 2B baseline)
2. Fine-tunes on MedQA at r=16 and re-evaluates all 57 subjects

The 2B baseline is saved separately as `E00_baseline_2B.csv` and also logged into `all_results.csv`  
under exp_id `E00_2B`.


In [ ]:
# ── Resume guard ─────────────────────────────────────────────────────────────
_skip_E12 = False
baseline_17b = {}  # will hold 2B baseline accs; used by E13 and seed repeats

baseline_17b_path = f"{WORKING_RESULTS_DIR}/E00_baseline_2B.csv"
if os.path.exists(baseline_17b_path):
    _b2 = pd.read_csv(baseline_17b_path)
    baseline_17b = dict(zip(_b2["subject"], _b2["accuracy"]))
    print(f"✅ 2B baseline loaded from Drive: {len(baseline_17b)} subjects")

if os.path.exists(RESULTS_CSV):
    _df_chk = safe_read_results()
    _done = _df_chk[(_df_chk["exp_id"]=="E12") & (_df_chk["subject"].isin(MMLU_SUBJECTS))]
    if len(_done) >= 50:
        print(f"✅ E12 already complete ({len(_done)} subjects). Skipping fine-tune.")
        _skip_E12 = True

if not _skip_E12:
    print("\n" + "="*60)
    print("▶  E12: Qwen3.5-2B, MedQA, rank=16, steps=500")
    print("="*60)

    # ── Step 1: 2B baseline (skip if already saved) ──────────────────────────
    if len(baseline_17b) < 50:
        print("📊 Running 2B baseline evaluation (57 MMLU subjects) ...")
        model_2b_base, tok_2b_base = load_model_4bit(MODEL_2B)
        model_2b_base.eval()
        baseline_17b = eval_all_mmlu(model_2b_base, tok_2b_base)

        # Save 2B baseline CSV
        pd.DataFrame([
            {"subject": s, "accuracy": a, "is_medical": s in MEDICAL_MMLU}
            for s, a in baseline_17b.items()
        ]).to_csv(baseline_17b_path, index=False)

        for subj, acc in baseline_17b.items():
            log_result("E00_2B", {
                "model": MODEL_2B, "domain": "none", "lora_rank": 0,
                "n_steps": 0, "replay_size": 0, "seed": SEED_PRIMARY,
                "subject": subj,
                "accuracy": round(acc, 4) if not np.isnan(acc) else "nan",
                "is_medical": subj in MEDICAL_MMLU,
                "delta_acc": 0, "forgetting": 0,
            })

        del model_2b_base, tok_2b_base
        torch.cuda.empty_cache(); gc.collect()
        print(f"✅ 2B baseline saved: mean={np.nanmean(list(baseline_17b.values())):.3f}")
    else:
        print(f"✅ 2B baseline already available: {len(baseline_17b)} subjects")

    # ── Step 2: fine-tune 2B on MedQA ────────────────────────────────────────
    ft_2b_E12, tok_2b_E12 = run_finetune(
        MODEL_2B, medqa_train, "E12", lora_rank=16, n_steps=500, seed=SEED_PRIMARY)
    ft_2b_E12.eval()

    post_accs_E12 = eval_all_mmlu(ft_2b_E12, tok_2b_E12)
    fgt_E12 = {}
    for subj in MMLU_SUBJECTS:
        base_acc = baseline_17b.get(subj, float("nan"))
        post_acc = post_accs_E12.get(subj, float("nan"))
        delta = post_acc - base_acc
        fgt_E12[subj] = delta
        log_result("E12", {
            "model": MODEL_2B, "domain": "medqa", "lora_rank": 16,
            "n_steps": 500, "replay_size": 0, "seed": SEED_PRIMARY,
            "subject": subj,
            "accuracy":  round(post_acc, 4) if not np.isnan(post_acc) else "nan",
            "is_medical": subj in MEDICAL_MMLU,
            "delta_acc":  round(delta, 4)    if not np.isnan(delta)    else "nan",
            "forgetting": round(-delta, 4)   if not np.isnan(delta)    else "nan",
        })

    prox_E12 = run_proximity_test("medqa", fgt_E12, "E12")
    log_result("E12_proximity", {
        "model": MODEL_2B, "domain": "medqa", "lora_rank": 16,
        "n_steps": 500, "replay_size": 0, "seed": SEED_PRIMARY,
        "subject": "ALL_PROXIMITY_STATS",
        **prox_E12, "delta_acc": 0, "forgetting": 0,
        "accuracy": 0, "is_medical": False,
    })

    print(f"\n📋 E12 SUMMARY (Qwen3.5-2B, r=16)")
    print(f"   Mean forgetting : {-np.nanmean(list(fgt_E12.values())):+.4f}")
    print(f"   H3 direction    : {prox_E12['h3_direction']}")
    print(f"   💾 Results saved → {RESULTS_CSV}")
    print(f"   ✅ E12 COMPLETE.")

    del ft_2b_E12, tok_2b_E12
    torch.cuda.empty_cache(); gc.collect()


### Cell 10 — E13: Qwen3.5-2B r=4 + H6 Spearman test

Fine-tunes the 2B model at r=4 to check whether the rank effect (H2) also replicates  
at smaller model size.

After E13 completes, the **H6 Spearman rank correlation** is computed between:  
- Per-subject forgetting from E02 (9B model, r=16)  
- Per-subject forgetting from E12 (2B model, r=16)  

If ρ > 0.70: the same subjects are most forgotten on both models → pattern is model-size invariant.  
If ρ < 0.50: the pattern is model-specific and this must be reported as a limitation.


In [ ]:
# ── Resume guard ─────────────────────────────────────────────────────────────
_skip_E13 = False
if os.path.exists(RESULTS_CSV):
    _df_chk = safe_read_results()
    _done = _df_chk[(_df_chk["exp_id"]=="E13") & (_df_chk["subject"].isin(MMLU_SUBJECTS))]
    if len(_done) >= 50:
        print(f"✅ E13 already complete ({len(_done)} subjects). Skipping.")
        _skip_E13 = True

if not _skip_E13:
    print("\n" + "="*60)
    print("▶  E13: Qwen3.5-2B, MedQA, rank=4, steps=500")
    print("="*60)

    # Load 2B baseline if not already in memory
    if len(baseline_17b) < 50:
        baseline_17b_path = f"{WORKING_RESULTS_DIR}/E00_baseline_2B.csv"
        if os.path.exists(baseline_17b_path):
            _b2 = pd.read_csv(baseline_17b_path)
            baseline_17b = dict(zip(_b2["subject"], _b2["accuracy"]))
        else:
            raise RuntimeError("❌ 2B baseline missing. Run Cell 9 (E12) first.")

    ft_2b_E13, tok_2b_E13 = run_finetune(
        MODEL_2B, medqa_train, "E13", lora_rank=4, n_steps=500, seed=SEED_PRIMARY)
    ft_2b_E13.eval()

    post_accs_E13 = eval_all_mmlu(ft_2b_E13, tok_2b_E13)
    fgt_E13 = {}
    for subj in MMLU_SUBJECTS:
        base_acc = baseline_17b.get(subj, float("nan"))
        post_acc = post_accs_E13.get(subj, float("nan"))
        delta = post_acc - base_acc
        fgt_E13[subj] = delta
        log_result("E13", {
            "model": MODEL_2B, "domain": "medqa", "lora_rank": 4,
            "n_steps": 500, "replay_size": 0, "seed": SEED_PRIMARY,
            "subject": subj,
            "accuracy":  round(post_acc, 4) if not np.isnan(post_acc) else "nan",
            "is_medical": subj in MEDICAL_MMLU,
            "delta_acc":  round(delta, 4)    if not np.isnan(delta)    else "nan",
            "forgetting": round(-delta, 4)   if not np.isnan(delta)    else "nan",
        })

    del ft_2b_E13, tok_2b_E13
    torch.cuda.empty_cache(); gc.collect()

# ── H6: Spearman correlation — 9B vs 2B per-subject forgetting ───────────────
print("\n" + "="*60)
print("📊 H6 — Generalisation test: Qwen3.5-9B vs Qwen3.5-2B")
print("="*60)

df_all = safe_read_results()

# Load 9B forgetting from E02
df_e02 = df_all[(df_all["exp_id"]=="E02")].copy()
df_e02["f"] = pd.to_numeric(df_e02["forgetting"], errors="coerce")
df_e02 = df_e02.dropna(subset=["f"])
fgt_9B = dict(zip(df_e02["subject"], df_e02["f"]))

# Load 2B forgetting from E12
df_e12 = df_all[(df_all["exp_id"]=="E12")].copy()
df_e12["f"] = pd.to_numeric(df_e12["forgetting"], errors="coerce")
df_e12 = df_e12.dropna(subset=["f"])
fgt_2B = dict(zip(df_e12["subject"], df_e12["f"]))

common_subj = [s for s in MMLU_SUBJECTS if s in fgt_9B and s in fgt_2B]

if len(common_subj) > 10:
    vals_9B = [fgt_9B[s] for s in common_subj]
    vals_2B = [fgt_2B[s] for s in common_subj]
    rho, rho_p = spearmanr(vals_9B, vals_2B)

    print(f"   n subjects compared : {len(common_subj)}")
    print(f"   Spearman ρ          : {rho:.3f}")
    print(f"   p-value             : {rho_p:.5f}")
    print(f"   Threshold           : ρ > 0.70 → pattern is model-size invariant")
    result_str = "✅ SUPPORTED" if rho > 0.70 else "❌ NOT SUPPORTED"
    print(f"   H6 result           : {result_str}")

    log_result("H6_spearman", {
        "model": "comparison_9B_vs_2B", "domain": "medqa",
        "lora_rank": 16, "n_steps": 500, "replay_size": 0,
        "seed": SEED_PRIMARY, "subject": "H6_GENERALISATION",
        "accuracy": 0, "is_medical": False, "delta_acc": 0, "forgetting": 0,
        "h6_spearman_rho": round(rho, 4),
        "h6_spearman_p":   round(rho_p, 6),
        "h6_n_subjects":   len(common_subj),
        "h6_result":       "supported" if rho > 0.70 else "not_supported",
    })
else:
    print(f"   ⚠️  Only {len(common_subj)} common subjects. Need E02 (Notebook A) + E12 both complete.")

print(f"\n✅ E12 + E13 + H6 COMPLETE. Safe to disconnect.")

download_results("after_E12_E13")


---
## Experiments E14–E17 — Seed Repeats (error bars)

Repeats the four primary conditions with `seed=7` instead of `seed=42`.  
This gives you **mean ± standard deviation** for all key results, which is required  
for a credible workshop paper submission.

| Experiment | Repeats | Seeds tested |
|---|---|---|
| E14 | E02 (9B, MedQA, r=16, no replay) | seed=7 |
| E15 | E01 (9B, MedQA, r=4, no replay) | seed=7 |
| E16 | E12 (2B, MedQA, r=16, no replay) | seed=7 |
| E17 | E10 (9B, MedQA, r=16, 100 real replay) | seed=7 |

Each seed repeat is run as a self-contained block with its own resume guard.  
You can run one per Colab session across multiple days.

⏱️ ~10 GPU-hours total (4 × ~2.5 hours)


### Cell 11 — E14–E17: seed repeats (one per Colab session recommended)


In [ ]:
SEED_REPEAT_EXPS = [
    # (exp_id, model_id, domain, rank, n_steps, replay_size, seed)
    ("E14", MODEL_9B, "medqa", 16, 500, 0,   SEED_REPEAT),  # repeat of E02
    ("E15", MODEL_9B, "medqa",  4, 500, 0,   SEED_REPEAT),  # repeat of E01
    ("E16", MODEL_2B, "medqa", 16, 500, 0,   SEED_REPEAT),  # repeat of E12
    ("E17", MODEL_9B, "medqa", 16, 500, 100, SEED_REPEAT),  # repeat of E10 (with replay)
]

for exp_id, model_id, domain, rank, n_steps, replay_size, seed in SEED_REPEAT_EXPS:

    # ── Resume guard ──────────────────────────────────────────────────────────
    _skip = False
    if os.path.exists(RESULTS_CSV):
        _df_chk = safe_read_results()
        _done = _df_chk[
            (_df_chk["exp_id"] == exp_id) &
            (_df_chk["subject"].isin(MMLU_SUBJECTS))
        ]
        if len(_done) >= 50:
            print(f"✅ {exp_id} already complete ({len(_done)} subjects). Skipping.")
            _skip = True

    if _skip:
        continue

    print(f"\n{'='*60}")
    print(f"▶  {exp_id}: model={model_id.split('/')[-1]} rank={rank} "
          f"steps={n_steps} replay={replay_size} seed={seed}")
    print(f"{'='*60}")

    # ── Build training data ───────────────────────────────────────────────────
    train_data = get_stratified_medqa(n_total=4000, seed=seed)
    if replay_size > 0:
        replay_buf = build_real_mmlu_replay(replay_size, seed=seed)
        train_data = concatenate_datasets([train_data, replay_buf]).shuffle(seed=seed)

    # ── Fine-tune ─────────────────────────────────────────────────────────────
    ft_model, ft_tok = run_finetune(model_id, train_data, exp_id,
                                    rank, n_steps, seed)
    ft_model.eval()

    # ── Evaluate ──────────────────────────────────────────────────────────────
    post_accs = eval_all_mmlu(ft_model, ft_tok)

    # Choose correct baseline for this model size
    # MODEL_9B = Qwen3.5-9B, MODEL_2B = Qwen3.5-2B
    baseline = baseline_accs if model_id == MODEL_9B else baseline_17b

    fgt = {}
    for subj in MMLU_SUBJECTS:
        base_acc = baseline.get(subj, float("nan"))
        post_acc = post_accs.get(subj, float("nan"))
        delta = post_acc - base_acc
        fgt[subj] = delta
        log_result(exp_id, {
            "model":       model_id,
            "domain":      domain,
            "lora_rank":   rank,
            "n_steps":     n_steps,
            "replay_size": replay_size,
            "seed":        seed,
            "subject":     subj,
            "accuracy":    round(post_acc, 4) if not np.isnan(post_acc) else "nan",
            "is_medical":  subj in MEDICAL_MMLU,
            "delta_acc":   round(delta, 4)    if not np.isnan(delta)    else "nan",
            "forgetting":  round(-delta, 4)   if not np.isnan(delta)    else "nan",
        })

    prox = run_proximity_test(domain, fgt, exp_id)
    log_result(f"{exp_id}_proximity", {
        "model": model_id, "domain": domain, "lora_rank": rank,
        "n_steps": n_steps, "replay_size": replay_size, "seed": seed,
        "subject": "ALL_PROXIMITY_STATS",
        **prox, "delta_acc": 0, "forgetting": 0,
        "accuracy": 0, "is_medical": False,
    })

    mean_fgt = -np.nanmean(list(fgt.values()))
    print(f"\n📋 {exp_id}: Mean fgt={mean_fgt:.4f} | H1 r={prox['h1_pearson_r']}")
    print(f"   💾 Results saved → {RESULTS_CSV}")
    print(f"   ✅ {exp_id} COMPLETE. Safe to disconnect.")

    del ft_model, ft_tok
    torch.cuda.empty_cache(); gc.collect()

download_results("after_seed_repeats")


---
## ✅ Notebook C complete

All results from E09–E17 are in `all_results.csv` on your Google Drive.

**Next step:** Run **Notebook D** (analysis + figures) on any machine — it only needs CPU  
and takes about 20 minutes. It reads `all_results.csv` and produces:
- Figure 1: 57×4 heat map (subjects × rank, coloured by forgetting)
- Figure 2: Proximity scatter (H1 Pearson r, 3 domains)
- Figure 3: Steps curve (H4)
- Figure 4: Replay comparison bar chart (H5)
- Figure 5: Rank effect + proximity direction (H2 + H3)
- Figure 6: 9B vs 2B per-subject scatter (H6)
- `FINAL_HYPOTHESIS_SUMMARY.csv` — hand this file back for paper writing


In [ ]:
print("\n" + "="*60)
print("✅ NOTEBOOK C COMPLETE — All experiments finished")
print("="*60)
print(f"\nAll results in: {RESULTS_CSV}")
print("→ Run Notebook D (analysis) on any machine to generate all figures")

# Print a quick completion summary
if os.path.exists(RESULTS_CSV):
    df_summary = safe_read_results()
    exp_counts = df_summary[df_summary["subject"].isin(MMLU_SUBJECTS)].groupby("exp_id").size()
    c_exps = [e for e in ["E09","E10","E11","E12","E13","E14","E15","E16","E17"]
              if e in exp_counts.index]
    print(f"\n📋 Notebook C experiments logged:")
    for e in c_exps:
        print(f"   {e}: {exp_counts[e]} subjects")
    missing = [e for e in ["E09","E10","E11","E12","E13","E14","E15","E16","E17"]
               if e not in exp_counts.index]
    if missing:
        print(f"   ⚠️  Not yet run: {missing}")
